# GPU vs CPU Performance

This notebook demonstrates the performance differences between CPU and GPU across three dimensions:

1. **Computation speed:** Matrix multiplications of increasing size on CPU vs GPU, showing when a GPU starts making a real difference.
2. **Model size & memory:** Loading Qwen3.5-0.8B and Qwen3.5-2B on each device, measuring how much RAM/VRAM each consumes and how inference time scales.
3. **Quantization trade-offs:** Running each model at 32 / 16 / 8 / 4-bit precision on both devices, tracking how memory usage, inference speed, and classification accuracy change.

All inference experiments use the same 10 rows from the [QEvasion](https://huggingface.co/datasets/ailsntua/QEvasion) dataset to keep results directly comparable.

> ⚠️ **This notebook requires a GPU runtime.** Enable it via: `Runtime → Change runtime type → T4 GPU (or higher)`.

## Environment Setup & Imports

In [1]:
# Libraries for quantized model loading and dataset access
!pip install -q bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00


In [2]:
# Imports
import re
import time
import torch
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed(SEED)

import warnings
warnings.filterwarnings("ignore", module="bitsandbytes")

In [3]:
# CPU info
ram_total = psutil.virtual_memory().total / (1024 ** 3)
print(f"CPU RAM: {ram_total:.1f} GB")

# GPU info
if torch.cuda.is_available():
  gpu_name = torch.cuda.get_device_name(0)
  vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
  print(f"GPU: {gpu_name}")
  print(f"VRAM: {vram_total:.1f} GB")
  print(f"CUDA: {torch.version.cuda}")
else:
  print("No GPU available, all experiments will run on CPU only.")

CPU RAM: 12.7 GB
GPU: Tesla T4
VRAM: 14.6 GB
CUDA: 12.8


In [4]:
!nvidia-smi

Sat Jun  6 08:53:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Memory Landscape

Before running any experiments, we establish a baseline: how much RAM (CPU memory) and VRAM (GPU memory) is available and how much is already in use. Every variable we create (in our case, tensors and models) lives on a specific device. If we use `torch.tensor(...)` it is stored in RAM, if we use `.to("cuda")` it moves to VRAM. We observe how memory grows as we allocate tensors and load models.

**Baseline memory.**

In [5]:
def print_memory_usage(tag: str) -> None:
  """
  Print current RAM and VRAM usage.

  Args:
      tag: a short description of the current state (e.g., 'Baseline')
  """
  ram_used = psutil.virtual_memory().used / (1024 ** 3)
  ram_total = psutil.virtual_memory().total / (1024 ** 3)
  print(f"{tag} → reserved RAM: {ram_used:.2f} / {ram_total:.1f} GB")

  if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated(0) / (1024 ** 3)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"{tag} → reserved VRAM: {vram_used:.4f} / {vram_total:.1f} GB")

In [6]:
print_memory_usage("Baseline, nothing allocated yet")

Baseline, nothing allocated yet → reserved RAM: 2.68 / 12.7 GB
Baseline, nothing allocated yet → reserved VRAM: 0.0000 / 14.6 GB


**Memory after allocating tensors.**

Each tensor value is `float32` by default (32 bits = 4 bytes). So a tensor of shape `(1000, 1000)` occupies: $1000 \times 1000 \times 4 \text{ bytes} = 4{,}000{,}000 \text{ bytes} \approx 3.81 \text{ MB}$.

In [7]:
# Allocate tensors on CPU
cpu_small = torch.randn(1000, 1000)
print(f"Small tensor on CPU → shape: {cpu_small.shape} | dtype: {cpu_small.dtype} | size: {cpu_small.nbytes / (1024**2):.2f} MB")
print_memory_usage("Memory allocation after the (1K x 1K) tensor on CPU")

cpu_large = torch.randn(10000, 10000)
print(f"\nBig tensor on CPU → shape: {cpu_large.shape} | dtype: {cpu_large.dtype} | size: {cpu_large.nbytes / (1024**2):.2f} MB")
print_memory_usage("Memory allocation after the (10K x 10K) tensor on CPU")

# Allocate tensors on GPU
gpu_small = torch.randn(1000, 1000, device="cuda")
print(f"\nSmall tensor on GPU → shape: {gpu_small.shape} | dtype: {gpu_small.dtype} | size: {gpu_small.nbytes / (1024**2):.2f} MB")
print_memory_usage("Memory allocation after the (1K x 1K) tensor on GPU")

gpu_large = torch.randn(10000, 10000, device="cuda")
print(f"\nBig tensor on GPU → shape: {gpu_large.shape} | dtype: {gpu_large.dtype} | size: {gpu_large.nbytes / (1024**2):.2f} MB")
print_memory_usage("Memory allocation after the (10K x 10K) tensor on GPU")

Small tensor on CPU → shape: torch.Size([1000, 1000]) | dtype: torch.float32 | size: 3.81 MB
Memory allocation after the (1K x 1K) tensor on CPU → reserved RAM: 2.68 / 12.7 GB
Memory allocation after the (1K x 1K) tensor on CPU → reserved VRAM: 0.0000 / 14.6 GB

Big tensor on CPU → shape: torch.Size([10000, 10000]) | dtype: torch.float32 | size: 381.47 MB
Memory allocation after the (10K x 10K) tensor on CPU → reserved RAM: 3.02 / 12.7 GB
Memory allocation after the (10K x 10K) tensor on CPU → reserved VRAM: 0.0000 / 14.6 GB

Small tensor on GPU → shape: torch.Size([1000, 1000]) | dtype: torch.float32 | size: 3.81 MB
Memory allocation after the (1K x 1K) tensor on GPU → reserved RAM: 3.11 / 12.7 GB
Memory allocation after the (1K x 1K) tensor on GPU → reserved VRAM: 0.0037 / 14.6 GB

Big tensor on GPU → shape: torch.Size([10000, 10000]) | dtype: torch.float32 | size: 381.47 MB
Memory allocation after the (10K x 10K) tensor on GPU → reserved RAM: 3.11 / 12.7 GB
Memory allocation after t

In [8]:
# Free everything so later sections start clean
del cpu_small, cpu_large, gpu_small, gpu_large
torch.cuda.empty_cache()
print_memory_usage("After releasing all tensors")

After releasing all tensors → reserved RAM: 2.75 / 12.7 GB
After releasing all tensors → reserved VRAM: 0.0000 / 14.6 GB


**Memory After Loading a Model.**

The 0.8B model has ~752 million parameters stored in `bfloat16` by default (16 bits = 2 bytes). So its memory footprint is approximately: $752{,}000{,}000 \times 2 \text{ bytes} = 1{,}504{,}000{,}000 \text{ bytes} \approx 1.40 \text{ GB}$.

In [9]:
MODEL_08B = "Qwen/Qwen3.5-0.8B"
MODEL_2B = "Qwen/Qwen3.5-2B"

In [10]:
def load_model(model_name: str, device: str, quantization_config: BitsAndBytesConfig | None = None, verbose: bool = True) -> tuple:
  """
  Load a model and its tokenizer onto a device, measuring time and memory.

  Args:
      model_name: Hugging Face model ID (e.g., 'Qwen/Qwen3.5-0.8B')
      device: 'cpu' or 'cuda'
      quantization_config: optional BitsAndBytesConfig for quantized loading
      verbose: if True, print model details (parameters, dtype)

  Returns:
      A tuple of (model, tokenizer, elapsed_seconds, memory_delta_gb)
  """
  # Memory before
  if device == "cuda":
    mem_before = torch.cuda.memory_allocated(0) / (1024 ** 3)
  else:
    mem_before = psutil.virtual_memory().used / (1024 ** 3)

  print_memory_usage(f"Before loading {model_name} on {device}")

  start = time.time()
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  if quantization_config is not None:
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quantization_config, device_map="auto")
  else:
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

  elapsed = time.time() - start

  # Memory after
  if device == "cuda":
    mem_after = torch.cuda.memory_allocated(0) / (1024 ** 3)
  else:
    mem_after = psutil.virtual_memory().used / (1024 ** 3)

  mem_delta = mem_after - mem_before

  print_memory_usage(f"After loading {model_name} on {device}")

  if verbose:
    print(f"\nLoad time: {elapsed:.2f}s")
    print(f"Memory used by model: {mem_delta:.4f} GB")

    total_params = sum(p.numel() for p in model.parameters())
    dtype = next(model.parameters()).dtype
    bytes_per_param = next(model.parameters()).element_size()
    expected_mb = total_params * bytes_per_param / (1024 ** 2)
    print(f"Model: {model_name}")
    print(f"Total parameters: {total_params:,} ({total_params / 1e9:.2f}B)")
    print(f"Dtype: {dtype} ({bytes_per_param} bytes/param)")
    if quantization_config is None:
      print(f"Expected memory: {total_params:,} x {bytes_per_param} = {expected_mb:,.1f} MB ({expected_mb / 1024:.2f} GB)\n")
    else:
      print(f"Actual memory: {mem_delta:.4f} GB (quantized: dtype reports compute type, not storage)")

  return model, tokenizer, elapsed, mem_delta

In [11]:
# Load 0.8B on CPU
model_08b_cpu, tokenizer_08b, time_08b_cpu, _ = load_model(MODEL_08B, "cpu")

del model_08b_cpu
print_memory_usage("After releasing 0.8B from CPU")

Before loading Qwen/Qwen3.5-0.8B on cpu → reserved RAM: 2.75 / 12.7 GB
Before loading Qwen/Qwen3.5-0.8B on cpu → reserved VRAM: 0.0000 / 14.6 GB


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-0.8B on cpu → reserved RAM: 1.89 / 12.7 GB
After loading Qwen/Qwen3.5-0.8B on cpu → reserved VRAM: 0.0000 / 14.6 GB

Load time: 30.28s
Memory used by model: -0.8646 GB
Model: Qwen/Qwen3.5-0.8B
Total parameters: 752,393,024 (0.75B)
Dtype: torch.bfloat16 (2 bytes/param)
Expected memory: 752,393,024 x 2 = 1,435.1 MB (1.40 GB)

After releasing 0.8B from CPU → reserved RAM: 1.89 / 12.7 GB
After releasing 0.8B from CPU → reserved VRAM: 0.0000 / 14.6 GB


In [12]:
# Load 0.8B on GPU
model_08b_gpu, _, time_08b_gpu, _ = load_model(MODEL_08B, "cuda")

del model_08b_gpu
torch.cuda.empty_cache()
print_memory_usage("After releasing 0.8B from GPU")

Before loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 1.89 / 12.7 GB
Before loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 0.0000 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 2.43 / 12.7 GB
After loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 1.4249 / 14.6 GB

Load time: 6.16s
Memory used by model: 1.4249 GB
Model: Qwen/Qwen3.5-0.8B
Total parameters: 752,393,024 (0.75B)
Dtype: torch.bfloat16 (2 bytes/param)
Expected memory: 752,393,024 x 2 = 1,435.1 MB (1.40 GB)

After releasing 0.8B from GPU → reserved RAM: 2.30 / 12.7 GB
After releasing 0.8B from GPU → reserved VRAM: 0.0000 / 14.6 GB


In [13]:
# Load 2B on CPU
model_2b_cpu, tokenizer_2b, time_2b_cpu, _ = load_model(MODEL_2B, "cpu")

del model_2b_cpu
print_memory_usage("After releasing 2B from CPU")

Before loading Qwen/Qwen3.5-2B on cpu → reserved RAM: 2.30 / 12.7 GB
Before loading Qwen/Qwen3.5-2B on cpu → reserved VRAM: 0.0000 / 14.6 GB


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-2B on cpu → reserved RAM: 3.37 / 12.7 GB
After loading Qwen/Qwen3.5-2B on cpu → reserved VRAM: 0.0000 / 14.6 GB

Load time: 80.85s
Memory used by model: 1.0632 GB
Model: Qwen/Qwen3.5-2B
Total parameters: 1,881,825,088 (1.88B)
Dtype: torch.bfloat16 (2 bytes/param)
Expected memory: 1,881,825,088 x 2 = 3,589.3 MB (3.51 GB)

After releasing 2B from CPU → reserved RAM: 3.37 / 12.7 GB
After releasing 2B from CPU → reserved VRAM: 0.0000 / 14.6 GB


In [14]:
# Load 2B on GPU
model_2b_gpu, _, time_2b_gpu, _ = load_model(MODEL_2B, "cuda")

del model_2b_gpu
torch.cuda.empty_cache()
print_memory_usage("After releasing 2B from GPU")

Before loading Qwen/Qwen3.5-2B on cuda → reserved RAM: 3.37 / 12.7 GB
Before loading Qwen/Qwen3.5-2B on cuda → reserved VRAM: 0.0000 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-2B on cuda → reserved RAM: 3.46 / 12.7 GB
After loading Qwen/Qwen3.5-2B on cuda → reserved VRAM: 3.5052 / 14.6 GB

Load time: 17.27s
Memory used by model: 3.5052 GB
Model: Qwen/Qwen3.5-2B
Total parameters: 1,881,825,088 (1.88B)
Dtype: torch.bfloat16 (2 bytes/param)
Expected memory: 1,881,825,088 x 2 = 3,589.3 MB (3.51 GB)

After releasing 2B from GPU → reserved RAM: 3.40 / 12.7 GB
After releasing 2B from GPU → reserved VRAM: 0.0000 / 14.6 GB


> **Why doesn't RAM drop back to its baseline after `del`?**
>
> Python's memory allocator reserves released memory for future reuse instead of returning it to the OS. So `psutil` still reports it as "used." VRAM, on the other hand, is managed by PyTorch's CUDA allocator and `empty_cache()` explicitly releases it back to the GPU driver, so it returns to near-zero.

## CPU vs GPU on Computation Speed

Matrix multiplication is the core operation inside every neural network layer, so by timing it in isolation on CPU and GPU, we measure the raw speed difference between the two devices.

**Small Matrix Multiplication.**

In [15]:
size = 256
iterations = 100

# CPU
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
for _ in range(iterations):
  mm_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start

# GPU (same matrices, moved to GPU)
a_gpu = a_cpu.to("cuda")
b_gpu = b_cpu.to("cuda")
print_memory_usage(f"Memory with {size}x{size} matrices on both devices")

# Warmup: one untimed run to initialize CUDA kernels
_ = a_gpu @ b_gpu

# GPU runs async: synchronize() ensures all operations finish before we read the clock
torch.cuda.synchronize()

start = time.time()
for _ in range(iterations):
  mm_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_time = time.time() - start

print(f"\nMatrix size: {size}x{size} | Iterations: {iterations}")
print(f"CPU time: {cpu_time:.4f}s")
print(f"GPU time: {gpu_time:.4f}s")
print(f"Speedup: {cpu_time / gpu_time:.1f}x")
print(f"CPU result sample: {mm_cpu[0, :5]}")
print(f"GPU result sample: {mm_gpu.cpu()[0, :5]}")
print(f"Results match: {torch.allclose(mm_cpu, mm_gpu.cpu())}") # First move mm_gpu to cpu, then compare
print(f"Results match (tolerance=0.01, as small differences are expected between CPU and GPU): {torch.allclose(mm_cpu, mm_gpu.cpu(), atol=1e-2)}")

Memory with 256x256 matrices on both devices → reserved RAM: 3.40 / 12.7 GB
Memory with 256x256 matrices on both devices → reserved VRAM: 0.0005 / 14.6 GB

Matrix size: 256x256 | Iterations: 100
CPU time: 0.1140s
GPU time: 0.0030s
Speedup: 38.2x
CPU result sample: tensor([  7.8040,  -8.5909, -24.0096,  22.1395, -10.8864])
GPU result sample: tensor([  7.8040,  -8.5909, -24.0096,  22.1395, -10.8864])
Results match: False
Results match (tolerance=0.01, as small differences are expected between CPU and GPU): True


**Large Matrix Multiplication.**

In [16]:
size = 4096
iterations = 100

# CPU
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
for _ in range(iterations):
  mm_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start

# GPU (same matrices, moved to GPU)
a_gpu = a_cpu.to("cuda")
b_gpu = b_cpu.to("cuda")
print_memory_usage(f"Memory with {size}x{size} matrices on both devices")

# Warmup: one untimed run to initialize CUDA kernels
_ = a_gpu @ b_gpu

torch.cuda.synchronize()
start = time.time()
for _ in range(iterations):
  mm_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_time = time.time() - start

print(f"\nMatrix size: {size}x{size} | Iterations: {iterations}")
print(f"CPU time: {cpu_time:.4f}s")
print(f"GPU time: {gpu_time:.4f}s")
print(f"Speedup: {cpu_time / gpu_time:.1f}x")
print(f"Results match (tolerance=0.01): {torch.allclose(mm_cpu, mm_gpu.cpu(), atol=1e-2)}")

Memory with 4096x4096 matrices on both devices → reserved RAM: 3.46 / 12.7 GB
Memory with 4096x4096 matrices on both devices → reserved VRAM: 0.1332 / 14.6 GB

Matrix size: 4096x4096 | Iterations: 100
CPU time: 130.0392s
GPU time: 3.8797s
Speedup: 33.5x
Results match (tolerance=0.01): True


In [17]:
# Free CPU and GPU
del a_cpu, b_cpu, mm_cpu, a_gpu, b_gpu, mm_gpu
torch.cuda.empty_cache()
print_memory_usage("After releasing large matrices")

After releasing large matrices → reserved RAM: 3.28 / 12.7 GB
After releasing large matrices → reserved VRAM: 0.0079 / 14.6 GB


> **Why does VRAM never reach exactly 0?**
>
> The ~8 MB remaining is the CUDA context, a driver-level memory allocated when GPU is first used. It persists for the entire session and is not released by `empty_cache()`.

## CPU vs GPU on LLM Inference

Now we move from raw matrix operations to real LLM inference. We load a small stratified sample from the QEvasion dataset, then run the same 10 rows through each model on CPU and GPU to compare inference time and accuracy.

**Prepare Sample Data.**

In [18]:
# Load the dataset
dataset = load_dataset("ailsntua/QEvasion", split="test")

print(f"Total rows: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Label distribution: {dataset.to_pandas()['clarity_label'].value_counts().to_dict()}")

README.md:   0%|          | 0.00/12.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/308 [00:00<?, ? examples/s]

Total rows: 308
Columns: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label']
Label distribution: {'Ambivalent': 206, 'Clear Reply': 79, 'Clear Non-Reply': 23}


In [19]:
# Parse to a dataframe
df = dataset.to_pandas()

# Stratified sample: 10 rows, proportional to class distribution
sample_df, _ = train_test_split(df, train_size=10, stratify=df["clarity_label"], random_state=SEED)

print(f"Sample size: {len(sample_df)}")
display(sample_df["clarity_label"].value_counts())

Sample size: 10


,count
clarity_label,
Ambivalent,7
Clear Reply,2
Clear Non-Reply,1


In [20]:
sample_df.head(3)

,title,date,president,url,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,annotator_id,annotator1,annotator2,annotator3,inaudible,multiple_questions,affirmative_questions,index,clarity_label,evasion_label
301,None,None,None,https://www.presidency.ucsb.edu/documents/the-...,3,"Q. Mr. President, a year ago when you were—had...","You know, I don't view— I just don't view life...",None,None,What does this say about the Democratic leader...,None,Deflection,Partial/half-answer,Partial/half-answer,False,False,False,301,Ambivalent,
87,None,None,None,https://www.presidency.ucsb.edu/documents/the-...,2,"Q. Mr. President, sorry, do you think it's wor...",It is absolutely working. We come across diffi...,None,None,Do you think it's working now the way it's go...,None,Explicit,Explicit,Explicit,False,False,False,87,Clear Reply,
300,None,None,None,https://www.presidency.ucsb.edu/documents/the-...,16,Q. Sorry. In your previous conversations with ...,On Iran?,None,None,Previous conversations with Prime Minister Ma...,None,Clarification,Clarification,Clarification,False,False,False,300,Clear Non-Reply,


**Helper functions.**

In [21]:
VALID_LABELS = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

In [22]:
SYSTEM_PROMPT = """You are a political response classifier.
Given a question and its corresponding answer, classify the answer into one of the following categories:
- Clear Reply
- Ambivalent
- Clear Non-Reply

Respond with the label only."""

In [23]:
def run_inference(model, tokenizer, sample_df: pd.DataFrame, device: str, max_new_tokens: int = 20) -> tuple[list[str], float, float]:
  """
  Run inference on sample rows and return predictions, time, and peak memory.

  Args:
      model: a loaded causal LM
      tokenizer: the matching tokenizer
      sample_df: dataframe with 'question' and 'answer' columns
      device: 'cpu' or 'cuda'
      max_new_tokens: maximum tokens to generate per row

  Returns:
      A tuple of (predictions list, elapsed seconds, peak memory in GB)
  """
  predictions = []

  if device == "cuda":
    torch.cuda.reset_peak_memory_stats(0)

  mem_before = psutil.virtual_memory().used / (1024 ** 3)
  print("")

  start = time.time()
  for i, row in sample_df.iterrows():
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {row['question']}\nAnswer: {row['interview_answer']}"},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
      outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    predictions.append(response)
    print(f"Row response {i}: {response} | Expected: {row['clarity_label']}")

  elapsed = time.time() - start

  if device == "cuda":
    peak_mem = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
  else:
    peak_mem = psutil.virtual_memory().used / (1024 ** 3) - mem_before

  print(f"\nTotal inference time: {elapsed:.2f}s")
  print(f"Peak memory during inference: {peak_mem:.4f} GB")

  return predictions, elapsed, peak_mem

In [24]:
def parse_response(response: str) -> str | None:
    """
    Extract a valid label from the model's raw output.

    Args:
        response: raw decoded text from the model

    Returns:
        One of the 3 valid labels, or None if no match is found
    """
    cleaned = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    cleaned = cleaned.strip()
    cleaned = cleaned.strip(".!?,;:\"'*`")
    return cleaned if cleaned in VALID_LABELS else None


def compute_accuracy(sample_df: pd.DataFrame, predictions: list[str]) -> float:
  """
  Compare predictions against gold labels and return accuracy.

  Args:
      sample_df: dataframe with 'clarity_label' column
      predictions: list of raw model outputs (one per row)

  Returns:
      Accuracy as a float between 0 and 1
  """
  parsed = [parse_response(p) for p in predictions]
  gold = sample_df["clarity_label"].tolist()
  correct = sum(p == g for p, g in zip(parsed, gold))
  total = len(gold)

  print(f"\nParsed: {parsed}")
  print(f"Gold: {gold}")
  print(f"Accuracy: {correct}/{total} = {correct / total:.2%}\n")

  return correct / total

### Qwen3.5-0.8B on CPU vs GPU

In [25]:
# Collect results from all experiments
results = []

In [26]:
# Load 0.8B on CPU
model, tokenizer, load_time, _ = load_model(MODEL_08B, "cpu", verbose=False)

# Run inference
predictions, inference_time, peak_mem = run_inference(model, tokenizer, sample_df, "cpu")

# Evaluate
accuracy = compute_accuracy(sample_df, predictions)

results.append({
    "Model": "Qwen3.5-0.8B",
    "Device": "CPU",
    "Bits": 16,
    "Bytes/Param": 2,
    "Load Time (s)": round(load_time, 2),
    "Peak Memory (GB)": round(peak_mem, 4),
    "Inference Time (s)": round(inference_time, 2),
    "Accuracy": round(accuracy, 4),
})

# Release
del model
print_memory_usage("After releasing 0.8B from CPU")

Before loading Qwen/Qwen3.5-0.8B on cpu → reserved RAM: 3.30 / 12.7 GB
Before loading Qwen/Qwen3.5-0.8B on cpu → reserved VRAM: 0.0079 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-0.8B on cpu → reserved RAM: 3.40 / 12.7 GB
After loading Qwen/Qwen3.5-0.8B on cpu → reserved VRAM: 0.0079 / 14.6 GB

Row response 301: Clear Reply | Expected: Ambivalent
Row response 87: Clear Non-Reply | Expected: Clear Reply
Row response 300: Ambivalent | Expected: Clear Non-Reply
Row response 63: Ambivalent | Expected: Clear Reply
Row response 269: Clear Non-Reply | Expected: Ambivalent
Row response 123: Ambivalent | Expected: Ambivalent
Row response 18: Ambivalent | Expected: Ambivalent
Row response 143: Clear Non-Reply | Expected: Ambivalent
Row response 104: Ambivalent | Expected: Ambivalent
Row response 9: Clear Non-Reply | Expected: Ambivalent

Total inference time: 229.49s
Peak memory during inference: 0.1896 GB

Parsed: ['Clear Reply', 'Clear Non-Reply', 'Ambivalent', 'Ambivalent', 'Clear Non-Reply', 'Ambivalent', 'Ambivalent', 'Clear Non-Reply', 'Ambivalent', 'Clear Non-Reply']
Gold: ['Ambivalent', 'Clear Reply', 'Clear Non-Reply', 'Clear Reply', '

In [27]:
# Load 0.8B on GPU
model, tokenizer, load_time, _ = load_model(MODEL_08B, "cuda", verbose=False)

# Run inference
predictions, inference_time, peak_mem = run_inference(model, tokenizer, sample_df, "cuda")

# Evaluate
accuracy = compute_accuracy(sample_df, predictions)

results.append({
    "Model": "Qwen3.5-0.8B",
    "Device": "GPU",
    "Bits": 16,
    "Bytes/Param": 2,
    "Load Time (s)": round(load_time, 2),
    "Peak Memory (GB)": round(peak_mem, 4),
    "Inference Time (s)": round(inference_time, 2),
    "Accuracy": round(accuracy, 4),
})

# Release
del model
torch.cuda.empty_cache()
print_memory_usage("After releasing 0.8B from GPU")

Before loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 3.59 / 12.7 GB
Before loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 0.0079 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 3.70 / 12.7 GB
After loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 1.4328 / 14.6 GB

Row response 301: Clear Reply | Expected: Ambivalent
Row response 87: Clear Non-Reply | Expected: Clear Reply
Row response 300: Ambivalent | Expected: Clear Non-Reply
Row response 63: Ambivalent | Expected: Clear Reply
Row response 269: Clear Non-Reply | Expected: Ambivalent
Row response 123: Ambivalent | Expected: Ambivalent
Row response 18: Ambivalent | Expected: Ambivalent
Row response 143: Clear Non-Reply | Expected: Ambivalent
Row response 104: Ambivalent | Expected: Ambivalent
Row response 9: Clear Non-Reply | Expected: Ambivalent

Total inference time: 6.95s
Peak memory during inference: 1.5424 GB

Parsed: ['Clear Reply', 'Clear Non-Reply', 'Ambivalent', 'Ambivalent', 'Clear Non-Reply', 'Ambivalent', 'Ambivalent', 'Clear Non-Reply', 'Ambivalent', 'Clear Non-Reply']
Gold: ['Ambivalent', 'Clear Reply', 'Clear Non-Reply', 'Clear Reply', '

### Qwen3.5-2B on CPU vs GPU

In [28]:
# Load 2B on CPU
model, tokenizer, load_time, _ = load_model(MODEL_2B, "cpu", verbose=False)

# Run inference
predictions, inference_time, peak_mem = run_inference(model, tokenizer, sample_df, "cpu")

# Evaluate
accuracy = compute_accuracy(sample_df, predictions)

results.append({
    "Model": "Qwen3.5-2B",
    "Device": "CPU",
    "Bits": 16,
    "Bytes/Param": 2,
    "Load Time (s)": round(load_time, 2),
    "Peak Memory (GB)": round(peak_mem, 4),
    "Inference Time (s)": round(inference_time, 2),
    "Accuracy": round(accuracy, 4),
})

# Release
del model
print_memory_usage("After releasing 2B from CPU")

Before loading Qwen/Qwen3.5-2B on cpu → reserved RAM: 3.89 / 12.7 GB
Before loading Qwen/Qwen3.5-2B on cpu → reserved VRAM: 0.0079 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-2B on cpu → reserved RAM: 4.06 / 12.7 GB
After loading Qwen/Qwen3.5-2B on cpu → reserved VRAM: 0.0079 / 14.6 GB

Row response 301: Clear Reply | Expected: Ambivalent
Row response 87: Clear Reply | Expected: Clear Reply
Row response 300: Clear Non-Reply | Expected: Clear Non-Reply
Row response 63: Clear Reply | Expected: Clear Reply
Row response 269: Clear Reply | Expected: Ambivalent
Row response 123: Clear Reply | Expected: Ambivalent
Row response 18: Clear Non-Reply | Expected: Ambivalent
Row response 143: Clear Reply | Expected: Ambivalent
Row response 104: Clear Reply | Expected: Ambivalent
Row response 9: Clear Reply | Expected: Ambivalent

Total inference time: 571.33s
Peak memory during inference: 0.2176 GB

Parsed: ['Clear Reply', 'Clear Reply', 'Clear Non-Reply', 'Clear Reply', 'Clear Reply', 'Clear Reply', 'Clear Non-Reply', 'Clear Reply', 'Clear Reply', 'Clear Reply']
Gold: ['Ambivalent', 'Clear Reply', 'Clear Non-Reply', 'Clear Reply', 'Ambivalent

In [29]:
# Load 2B on GPU
model, tokenizer, load_time, _ = load_model(MODEL_2B, "cuda", verbose=False)

# Run inference
predictions, inference_time, peak_mem = run_inference(model, tokenizer, sample_df, "cuda")

# Evaluate
accuracy = compute_accuracy(sample_df, predictions)

results.append({
    "Model": "Qwen3.5-2B",
    "Device": "GPU",
    "Bits": 16,
    "Bytes/Param": 2,
    "Load Time (s)": round(load_time, 2),
    "Peak Memory (GB)": round(peak_mem, 4),
    "Inference Time (s)": round(inference_time, 2),
    "Accuracy": round(accuracy, 4),
})

# Release
del model
torch.cuda.empty_cache()
print_memory_usage("After releasing 2B from GPU")

Before loading Qwen/Qwen3.5-2B on cuda → reserved RAM: 4.24 / 12.7 GB
Before loading Qwen/Qwen3.5-2B on cuda → reserved VRAM: 0.0079 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-2B on cuda → reserved RAM: 4.29 / 12.7 GB
After loading Qwen/Qwen3.5-2B on cuda → reserved VRAM: 3.5131 / 14.6 GB

Row response 301: Clear Reply | Expected: Ambivalent
Row response 87: Clear Reply | Expected: Clear Reply
Row response 300: Clear Non-Reply | Expected: Clear Non-Reply
Row response 63: Clear Reply | Expected: Clear Reply
Row response 269: Clear Reply | Expected: Ambivalent
Row response 123: Clear Reply | Expected: Ambivalent
Row response 18: Clear Reply | Expected: Ambivalent
Row response 143: Clear Reply | Expected: Ambivalent
Row response 104: Clear Reply | Expected: Ambivalent
Row response 9: Clear Reply | Expected: Ambivalent

Total inference time: 9.08s
Peak memory during inference: 3.6293 GB

Parsed: ['Clear Reply', 'Clear Reply', 'Clear Non-Reply', 'Clear Reply', 'Clear Reply', 'Clear Reply', 'Clear Reply', 'Clear Reply', 'Clear Reply', 'Clear Reply']
Gold: ['Ambivalent', 'Clear Reply', 'Clear Non-Reply', 'Clear Reply', 'Ambivalent', 'Ambi

## 6. Quantization: Memory vs Accuracy Trade-off

Quantization reduces the number of bits used to store each model parameters. A 32-bit model stores each weight as a 4-byte float, while a 4-bit model compresses it down to half a byte. This dramatically reduces memory usage, making it possible to run larger models on limited hardware. The trade-off is that lower precision may reduce accuracy.

We test Qwen3.5-0.8B at 32, 16, 8, and 4-bit precision on GPU, measuring memory usage, inference time, and classification accuracy.

In [30]:
# 32-bit (float32)
model, tokenizer, load_time, mem_delta = load_model(MODEL_08B, "cuda", quantization_config=None, verbose=False)

# Force float32 (default is bfloat16)
mem_before = torch.cuda.memory_allocated(0) / (1024 ** 3)
model = model.float()
mem_after = torch.cuda.memory_allocated(0) / (1024 ** 3)
mem_delta_f32 = mem_after - mem_before + mem_delta # Total memory = loading in bfloat16 (mem_delta) + converting to float32

print_memory_usage("0.8B loaded in float32")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nLoad time: {load_time:.2f}s")
print(f"Memory used by model: {mem_delta_f32:.4f} GB")
print(f"Model: {MODEL_08B}")
print(f"Total parameters: {total_params:,} ({total_params / 1e9:.2f}B)")
print(f"Dtype: {next(model.parameters()).dtype} (4 bytes/param)")
print(f"Expected memory: {total_params:,} x 4 = {total_params * 4 / (1024**2):,.1f} MB ({total_params * 4 / (1024**3):.2f} GB)")

predictions, inference_time, peak_mem = run_inference(model, tokenizer, sample_df, "cuda")
accuracy = compute_accuracy(sample_df, predictions)

results.append({
    "Model": "Qwen3.5-0.8B",
    "Device": "GPU",
    "Bits": 32,
    "Bytes/Param": 4,
    "Load Time (s)": round(load_time, 2),
    "Peak Memory (GB)": round(peak_mem, 4),
    "Inference Time (s)": round(inference_time, 2),
    "Accuracy": round(accuracy, 4),
})

del model
torch.cuda.empty_cache()
print_memory_usage("After releasing 0.8B float32")

Before loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 4.25 / 12.7 GB
Before loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 0.0079 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 4.27 / 12.7 GB
After loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 1.4328 / 14.6 GB
0.8B loaded in float32 → reserved RAM: 4.19 / 12.7 GB
0.8B loaded in float32 → reserved VRAM: 2.8108 / 14.6 GB

Load time: 9.33s
Memory used by model: 2.8029 GB
Model: Qwen/Qwen3.5-0.8B
Total parameters: 752,393,024 (0.75B)
Dtype: torch.float32 (4 bytes/param)
Expected memory: 752,393,024 x 4 = 2,870.2 MB (2.80 GB)

Row response 301: Clear Reply | Expected: Ambivalent
Row response 87: Clear Non-Reply | Expected: Clear Reply
Row response 300: Ambivalent | Expected: Clear Non-Reply
Row response 63: Ambivalent | Expected: Clear Reply
Row response 269: Clear Non-Reply | Expected: Ambivalent
Row response 123: Ambivalent | Expected: Ambivalent
Row response 18: Ambivalent | Expected: Ambivalent
Row response 143: Clear Non-Reply | Expected: Ambivalent
Row response 104: Ambivalent | Expected: Ambivalent
Row response 9: Clear Non-Reply | Expected: Ambiv

> **16-bit (bfloat16):** Already measured in the subsection of Qwen3.5-0.8B on GPU.

In [31]:
# 8-bit quantization
config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model, tokenizer, load_time, _ = load_model(MODEL_08B, "cuda", quantization_config=config_8bit)

predictions, inference_time, peak_mem = run_inference(model, tokenizer, sample_df, "cuda")
accuracy = compute_accuracy(sample_df, predictions)

results.append({
    "Model": "Qwen3.5-0.8B",
    "Device": "GPU",
    "Bits": 8,
    "Bytes/Param": 1,
    "Load Time (s)": round(load_time, 2),
    "Peak Memory (GB)": round(peak_mem, 4),
    "Inference Time (s)": round(inference_time, 2),
    "Accuracy": round(accuracy, 4),
})

del model
torch.cuda.empty_cache()
print_memory_usage("After releasing 0.8B 8-bit")

Before loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 4.18 / 12.7 GB
Before loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 0.0079 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 5.38 / 12.7 GB
After loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 0.9644 / 14.6 GB

Load time: 13.42s
Memory used by model: 0.9565 GB
Model: Qwen/Qwen3.5-0.8B
Total parameters: 752,393,024 (0.75B)
Dtype: torch.bfloat16 (2 bytes/param)
Actual memory: 0.9565 GB (quantized: dtype reports compute type, not storage)

Row response 301: Clear Non-Reply | Expected: Ambivalent
Row response 87: Clear Non-Reply | Expected: Clear Reply
Row response 300: Ambivalent | Expected: Clear Non-Reply
Row response 63: Ambivalent | Expected: Clear Reply
Row response 269: Clear Non-Reply | Expected: Ambivalent
Row response 123: Clear Non-Reply | Expected: Ambivalent
Row response 18: Ambivalent | Expected: Ambivalent
Row response 143: Clear Non-Reply | Expected: Ambivalent
Row response 104: Ambivalent | Expected: Ambivalent
Row response 9: Clear Non-Reply | Expected: Ambivalent

Total inference time: 14.85s
Peak memory during inference: 1.0726 GB

P

In [32]:
# 4-bit quantization
config_4bit = BitsAndBytesConfig(load_in_4bit=True)

model, tokenizer, load_time, _ = load_model(MODEL_08B, "cuda", quantization_config=config_4bit)

predictions, inference_time, peak_mem = run_inference(model, tokenizer, sample_df, "cuda")
accuracy = compute_accuracy(sample_df, predictions)

results.append({
    "Model": "Qwen3.5-0.8B",
    "Device": "GPU",
    "Bits": 4,
    "Bytes/Param": 0.5,
    "Load Time (s)": round(load_time, 2),
    "Peak Memory (GB)": round(peak_mem, 4),
    "Inference Time (s)": round(inference_time, 2),
    "Accuracy": round(accuracy, 4),
})

del model
torch.cuda.empty_cache()
print_memory_usage("After releasing 0.8B 4-bit")

Before loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 5.36 / 12.7 GB
Before loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 0.0089 / 14.6 GB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

After loading Qwen/Qwen3.5-0.8B on cuda → reserved RAM: 4.08 / 12.7 GB
After loading Qwen/Qwen3.5-0.8B on cuda → reserved VRAM: 0.7515 / 14.6 GB

Load time: 5.73s
Memory used by model: 0.7426 GB
Model: Qwen/Qwen3.5-0.8B
Total parameters: 503,585,600 (0.50B)
Dtype: torch.bfloat16 (2 bytes/param)
Actual memory: 0.7426 GB (quantized: dtype reports compute type, not storage)

Row response 301: Clear Non-Reply | Expected: Ambivalent
Row response 87: Clear Non-Reply | Expected: Clear Reply
Row response 300: Clear Non-Reply | Expected: Clear Non-Reply
Row response 63: Clear Non-Reply | Expected: Clear Reply
Row response 269: Clear Non-Reply | Expected: Ambivalent
Row response 123: Clear Non-Reply | Expected: Ambivalent
Row response 18: Clear Non-Reply | Expected: Ambivalent
Row response 143: Clear Non-Reply | Expected: Ambivalent
Row response 104: Clear Non-Reply | Expected: Ambivalent
Row response 9: Clear Non-Reply | Expected: Ambivalent

Total inference time: 16.85s
Peak memory during infe

## Final Summary

In [33]:
# All results collected throughout the notebook in one table
results_df = pd.DataFrame(results)
display(results_df.style.hide(axis="index").format(precision=2))

Model,Device,Bits,Bytes/Param,Load Time (s),Peak Memory (GB),Inference Time (s),Accuracy
Qwen3.5-0.8B,CPU,16,2.00,4.50,0.19,229.49,0.30
Qwen3.5-0.8B,GPU,16,2.00,5.21,1.54,6.95,0.30
Qwen3.5-2B,CPU,16,2.00,4.32,0.22,571.33,0.30
Qwen3.5-2B,GPU,16,2.00,5.35,3.63,9.08,0.30
Qwen3.5-0.8B,GPU,32,4.00,9.33,2.95,5.01,0.30
Qwen3.5-0.8B,GPU,8,1.00,13.42,1.07,14.85,0.20
Qwen3.5-0.8B,GPU,4,0.50,5.73,0.86,16.85,0.10


**Key Takeaways**

- **GPU is dramatically faster for inference:** 0.8B runs ~33x faster on GPU than CPU (6.95s vs 229s). For 2B the gap is ~63x (9.08s vs 571s).
- **Larger models are much slower on CPU:** The 0.8B model takes 229s on CPU, the 2B takes 571s.
- **32-bit wastes memory for no gain:** Float32 uses ~2x the VRAM of bfloat16 (2.95 vs 1.54 GB) with the same accuracy (0.30). Bfloat16 is the practical default.
- **Quantization saves memory but hurts accuracy:** 8-bit cuts VRAM to 1.07 GB but accuracy drops from 0.30 to 0.20. At 4-bit, VRAM drops to 0.86 GB but accuracy falls to 0.10.
- **Quantization does not speed up inference:** 8-bit (14.85s) and 4-bit (16.85s) are slower than 16-bit (6.95s) due to dequantization overhead during computation.
- **CPU peak memory is unreliable:** The -0.01 GB value for CPU shows that `psutil` cannot precisely track per-model memory on CPU the way CUDA can on GPU.
- **The sweet spot for this model is bfloat16 on GPU:** best balance of memory, speed, and accuracy.